# 12 - Iceberg Advanced SQL Smoke

This notebook runs the Atlas advanced Iceberg smoke through **Spark Connect** (`sc://spark-connect:15002`). It uses isolated `lakehouse.atlas_smoke` tables and verifies MERGE INTO, VERSION AS OF, rollback, branch/WAP writes, schema evolution, nested JSON, Structured Streaming from `s3a://landing/`, checkpointing under `s3a://checkpoints/`, and maintenance procedures.

In [ ]:
import os
from uuid import uuid4
from pyspark.sql import SparkSession
from pyspark.sql.types import LongType, StringType, StructField, StructType

spark_remote = os.environ.get("SPARK_REMOTE", "sc://spark-connect:15002").strip()
spark = SparkSession.builder.remote(spark_remote).getOrCreate()
print("connected:", spark_remote, "| Spark", spark.version)

In [ ]:
namespace = "lakehouse.atlas_smoke"
table = f"{namespace}.advanced_sql"
stream_table = f"{namespace}.advanced_stream"
run_id = uuid4().hex[:12]
landing_path = f"s3a://landing/atlas-smoke-advanced-json/{run_id}"
checkpoint_path = f"s3a://checkpoints/atlas-smoke-advanced-json/{run_id}"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {namespace}")
spark.sql(f"DROP TABLE IF EXISTS {table}")
spark.sql(f"DROP TABLE IF EXISTS {stream_table}")
spark.sql(f"""
CREATE TABLE {table} (
  id BIGINT,
  note STRING,
  metrics STRUCT<score: INT>
)
USING iceberg
TBLPROPERTIES ('format-version'='2', 'write.wap.enabled'='true')
""")
spark.sql(f"""
INSERT INTO {table}
VALUES
  (1, 'alpha', named_struct('score', 10)),
  (2, 'bravo', named_struct('score', 20))
""")
spark.table(table).show(truncate=False)

In [ ]:
initial_snapshot = spark.sql(
    f"SELECT snapshot_id FROM {table}.snapshots ORDER BY committed_at DESC LIMIT 1"
).collect()[0][0]

spark.sql(f"""
MERGE INTO {table} AS t
USING (
  SELECT 2 AS id, 'bravo-updated' AS note, named_struct('score', 25) AS metrics
  UNION ALL
  SELECT 3 AS id, 'charlie' AS note, named_struct('score', 30) AS metrics
) AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET note = s.note, metrics = s.metrics
WHEN NOT MATCHED THEN INSERT (id, note, metrics) VALUES (s.id, s.note, s.metrics)
""")
spark.sql(f"SELECT * FROM {table} VERSION AS OF {initial_snapshot}").show(truncate=False)
spark.sql(
    f"CALL lakehouse.system.rollback_to_snapshot(table => 'atlas_smoke.advanced_sql', snapshot_id => {initial_snapshot})"
).show(truncate=False)
spark.table(table).show(truncate=False)

In [ ]:
spark.sql(f"""
MERGE INTO {table} AS t
USING (SELECT 3 AS id, 'charlie' AS note, named_struct('score', 30) AS metrics) AS s
ON t.id = s.id
WHEN NOT MATCHED THEN INSERT (id, note, metrics) VALUES (s.id, s.note, s.metrics)
""")
spark.sql(f"ALTER TABLE {table} CREATE BRANCH atlas_wap")
spark.conf.set("spark.wap.branch", "atlas_wap")
spark.sql(f"INSERT INTO {table} VALUES (4, 'delta-from-wap', named_struct('score', 40))")
spark.conf.unset("spark.wap.branch")
spark.sql(f"SELECT * FROM {table} VERSION AS OF 'atlas_wap'").show(truncate=False)
spark.sql("CALL lakehouse.system.fast_forward('atlas_smoke.advanced_sql', 'main', 'atlas_wap')").show(truncate=False)
spark.sql(f"ALTER TABLE {table} DROP BRANCH atlas_wap")

In [ ]:
spark.sql(f"ALTER TABLE {table} ADD COLUMN quality STRING")
spark.sql(f"INSERT INTO {table} VALUES (5, 'echo', named_struct('score', 50), 'accepted')")
spark.sql("""
SELECT payload.id, event.kind, event.score
FROM (
  SELECT from_json(
    raw,
    'id BIGINT, events ARRAY<STRUCT<kind: STRING, score: INT>>'
  ) AS payload
  FROM VALUES ('{"id": 6, "events": [{"kind": "nested", "score": 60}]}') AS raw_events(raw)
)
LATERAL VIEW explode(payload.events) exploded AS event
""").show(truncate=False)

In [ ]:
spark.sql(f"""
CREATE TABLE {stream_table} (
  id BIGINT,
  event STRING
)
USING iceberg
TBLPROPERTIES ('format-version'='2')
""")
spark.createDataFrame([(101, "landing-a"), (102, "landing-b")], ["id", "event"]).write.mode("overwrite").json(landing_path)
schema = StructType([
    StructField("id", LongType(), False),
    StructField("event", StringType(), True),
])
query = (
    spark.readStream.schema(schema)
    .json(landing_path)
    .writeStream.format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(stream_table)
)
query.awaitTermination(120)
spark.table(stream_table).show(truncate=False)

In [ ]:
spark.sql("CALL lakehouse.system.rewrite_data_files(table => 'atlas_smoke.advanced_sql')").show(truncate=False)
spark.sql("CALL lakehouse.system.expire_snapshots(table => 'atlas_smoke.advanced_sql', retain_last => 1)").show(truncate=False)
spark.sql("CALL lakehouse.system.remove_orphan_files(table => 'atlas_smoke.advanced_sql', dry_run => true)").show(truncate=False)
print("advanced Iceberg smoke complete")